> **Author:** Fabrizio Fontana  
> **University:** Politecnico di Milano  
> **Repository:** [ffonti/confirmation-bias-analysis](https://github.com/ffonti/confirmation-bias-analysis)  
> **Supervisor:** Prof. Cinzia Cappiello  
> **Co-supervisor:** Dott. Mattia Sabella

# **NLI Evaluation**
This notebook measures the *Confirmation Bias* using the `deberta-v3-large` model as a Cross-Encoder to derive Entailment metrics.

In [1]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.nli import compute_nli_metrics

## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [ ]:
# Define the path to the generated results (JSONL format)
# file_path = "../data/interim/3_fever_deepseek_r1_results.jsonl"
# file_path = "../data/interim/4_truthfulqa_deepseek_r1_results.jsonl"
file_path = "../data/interim/5_mmlu_pro_gpt_4o_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
except FileNotFoundError:
    print(f"File not found at the requested path ({file_path}).")

Loaded 2 samples.


## **Evaluation**
Compute the evaluation metrics by sending the generated results to the model, that processes the Entailment Score.

In [3]:
# Compute the NLI metrics
df_evaluated = compute_nli_metrics(df_results)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [4]:
# Aggregation of the average scores for each metric
mean_combined = df_evaluated["cb_combined"].mean()
mean_shift = df_evaluated["cb_shift"].mean()
mean_self = df_evaluated["cb_self"].mean()

print("Results of the NLI Evaluation:")
print(f"Confirmation Bias Mean (Combined):  {mean_combined:.6f}")
print(f"Average Stance Shift Drop:          {mean_shift:.6f}")
print(f"Average Self-Contradiction Drop:    {mean_self:.6f}")

display(df_evaluated[["sample", "cb_shift", "cb_self", "cb_combined"]].head(10))

Results of the NLI Evaluation:
Confirmation Bias Mean (Combined):  0.035901
Average Stance Shift Drop:          0.186145
Average Self-Contradiction Drop:    -0.114343


,sample,cb_shift,cb_self,cb_combined
0,1,0.487104,0.007976,0.247540
1,2,-0.114815,-0.236662,-0.175738


## **Exporting**
Export the results to a CSV file for later use in the final analysis notebook.

In [5]:
# Extract filename from the input path
base_name = os.path.basename(file_path).replace("_results.jsonl", "")
output_file = f"../data/interim/{base_name}_nli.csv"

# Save the evaluated DataFrame to the interim data directory
df_evaluated.to_csv(output_file, index=False)
print(f"Saved evaluation file to {output_file}")

Saved evaluation file to ../data/interim/5_mmlu_pro_deepseek_r1_nli.csv
